In [1]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

d:\Coding\Genarative-AI\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
docs = (
    PyPDFLoader('.\Document\Company_Policies.pdf').load() +
    PyPDFLoader('.\Document\Company_Profile.pdf').load() +
    PyPDFLoader('.\Document\Product_and_Pricing.pdf').load()
    
)

<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:3: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:3: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
C:\Users\MARUF HASAN\AppData\Local\Temp\ipykernel_2216\2514151863.py:2: SyntaxWarning: invalid escape sequence '\D'
  PyPDFLoader('.\Document\Company_Policies.pdf').load() +
C:\Users\MARUF HASAN\AppData\Local\Temp\ipykernel_2216\2514151863.py:3: SyntaxWarning: invalid escape sequence '\D'
  PyPDFLoader('.\Document\Company_Profile.pdf').load() +
C:\Users\MARUF HASAN\AppData\Local\Temp\ipykernel_2216\2514151863.py:4: SyntaxWarning: invalid escape sequence '\D'
  PyPDFLoader('.\Document\Product_and_Pricing.pdf').load()


In [3]:
chunks = RecursiveCharacterTextSplitter(chunk_size = 600,chunk_overlap = 150).split_documents(docs)
print(len(chunks))

16


In [4]:
embed_model = HuggingFaceEmbeddings(model= "sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1236.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
vector_store = FAISS.from_documents(chunks,embed_model)
retriever = vector_store.as_retriever(search_type = 'similarity',search_kwargs={'k':4})

In [8]:
import os
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

In [42]:
class State(TypedDict):
    question:str
    need_retrieval:bool

    docs:list[Document]
    answer:str

In [43]:
from langchain_core.output_parsers import StrOutputParser
class RetrieveDecision(BaseModel):
    should_retrieve:bool = Field(
        ...,
        description = "True if external documents are needed to answer reliably, else False."
    )

decide_retrieval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You decide whether retrieval is needed.\n"
            "Return JSON that matches this schema:\n"
            "{{'should_retrieve': boolean}}\n\n"
            "Guidelines:\n"
            "- should_retrieve=True if answering requires specific facts, citations, or info likely not in the model.\n"
            "- should_retrieve=False for general explanations, definitions, or reasoning that doesn't need sources.\n"
            "- If unsure, choose True."
        ),
        ("human", "Question: {question}")
    ]
)

retrieve_decied_chain = decide_retrieval_prompt | llm.with_structured_output(
    RetrieveDecision,
    method="json_mode"  
)
def decide_revrieval_node(state: "State"):
    decision:RetrieveDecision = retrieve_decied_chain.invoke({'question':state['question']})

    return {'need_retrieval':decision.should_retrieve}

In [44]:
generation_prompt = ChatPromptTemplate.from_messages([
     (
            "system",
            "Answer the question using only your general knowledge.\n"
            "Do NOT assume access to external documents.\n"
            "If you are unsure or the answer requires specific sources, say:\n"
            "'I don't know based on my general knowledge.'"
        ),
        ("human", "{question}")]
)

genaration_chain = generation_prompt | llm | StrOutputParser()

def generation_node(state:State):
    out = genaration_chain.invoke({'question':state['question']})
    return {'answer':out}

In [54]:
def retrieve_node(state:State):
    return {'docs':retriever.invoke(state['question'])}

In [55]:
def route_after_decide(state:State) -> Literal['generate_direct','retrieve']:
    if state['need_retrieval']:
        return 'retrieve'
    return 'generate_direct'


In [56]:
g = StateGraph(State)

# --------------------
# Nodes
# --------------------
g.add_node("decide_retrieval", decide_revrieval_node)
g.add_node("generate_direct", generation_node)
g.add_node("retrieve", retrieve_node)

# --------------------
# Edges
# --------------------
g.add_edge(START, "decide_retrieval")

g.add_conditional_edges(
    "decide_retrieval",
    route_after_decide,
    {
        "generate_direct": "generate_direct",
        "retrieve": "retrieve",
    },
)

g.add_edge("generate_direct", END)
g.add_edge("retrieve", END)  # temporary END for retrieval path

app = g.compile()


In [57]:
result = app.invoke(
    {
        "question": "Who is the ceo of nexa ai",
        "need_retrieval": False,
        "docs": [],
        "answer": "",
    }
)

# পুরো result প্রিন্ট করে দেখি কি কি আছে
print("Final State:")
print(result)

# print(result["answer"])  <-- এটি আপাতত কমেন্ট করে রাখুন


Final State:
{'question': 'Who is the ceo of nexa ai', 'need_retrieval': True, 'docs': [Document(id='c9eea37b-b490-4fd1-a25a-2f821fad11b8', metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-14T13:07:17+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-14T13:07:17+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '.\\Document\\Company_Profile.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='Founder\nAarav Mehta founded NexaAI after over 10 years of experience in enterprise data platforms and\ncloud infrastructure.\nHe previously worked with global consulting firms where he led multiple large-scale digital\ntransformation projects.\nLeadership Team\nThe leadership team brings experience across AI engineering, product management, and business\noperations.\n\x7f\nAarav Mehta – CEO & Founder\n\x7f\nRiya Kapoor – CTO (Distributed systems & AI p

In [58]:
print(result['answer'])